# Improved Mean Flows (iMF): 1-NFE Class-Conditional Synthesis & Flexible CFG

This self-contained CPU notebook demonstrates **Improved Mean Flows (iMF)**
(Geng et al., CVPR 2026) implemented in
`swirl_dynamics.projects.distillation.improved_mean_flow` on top of
`swirl_dynamics.lib.flow_map`:

1. **Variance Elimination Oracle:** Empirically contrasts the compound-velocity
   JVP variance of original MeanFlow (noisy conditional tangent $x_1 - x_0$)
   against iMF (deterministic marginal tangent $v_\theta(x_t, t)$):
   $$
   V_\theta(x_t, t, s) = u_\theta(x_t, t, s) + (t - s)\thinspace\operatorname{sg}\Big(\partial_x u_\theta\thinspace\operatorname{sg}(v_\theta) + \partial_t u_\theta\Big).
   $$
2. **Auxiliary $v$-Head Training:** Trains `InContextMeanFlowTransformer`
   with `ConditionalImprovedMeanFlowModel(use_aux_v_head=True, cfg_mode="interval")`
   and `ImprovedMeanFlowTrainer` on a synthetic 1-D class-conditional periodic
   waveform dataset.
3. **1-NFE Flexible CFG Sweeps:** Evaluates a single checkpoint across guidance
   scales $\omega \in \lbrace 1, 2, 4 \rbrace$ and active guidance intervals
   $[t_{\min}, t_{\max}]$ in a single network evaluation (1 NFE) using only
   `head="u"`.


## 1. Installation & Imports


In [ ]:
!pip install -q "flax>=0.12.8" git+https://github.com/google-research/swirl-dynamics.git

In [ ]:
import os
import tempfile

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
from swirl_dynamics.lib import flow_map
from swirl_dynamics.projects.distillation.improved_mean_flow import backbones
from swirl_dynamics.projects.distillation.improved_mean_flow import models
from swirl_dynamics.projects.distillation.improved_mean_flow import trainers
from swirl_dynamics.projects.distillation.improved_mean_flow.configs import config_1d

## 2. Configuration


In [ ]:
config = config_1d.get_config()
workdir = tempfile.mkdtemp(prefix="imf_demo_")

## 3. MF vs. iMF Compound-Velocity Variance Comparison

For independent $x_0, x_1 \sim \mathcal{N}(0, I)$ with $x_t = (1 - t) x_0 + t x_1$,
the marginal standard deviation is $\sigma_t = \sqrt{(1 - t)^2 + t^2}$, the
marginal velocity is $v^*(x_t, t) = \frac{2t - 1}{\sigma_t^2} x_t$, and the exact
average velocity is $u^*(x_t, t, s) = \frac{1 - \sigma_s / \sigma_t}{t - s} x_t$.
Evaluating `flow_map.compute_compound_velocity` with tangent $v^*(x_t, t)$
eliminates conditional JVP variance completely.


In [ ]:
rng_oracle = jax.random.PRNGKey(config.seed)
k0, k1 = jax.random.split(rng_oracle)
x0_g = jax.random.normal(k0, (128, 4, 1))
x1_g = jax.random.normal(k1, (128, 4, 1))
t_g = jnp.full((128,), 0.7)
s_g = jnp.full((128,), 0.2)
xt_g = (1.0 - t_g[:, None, None]) * x0_g + t_g[:, None, None] * x1_g


def _exact_u(x_in, t_in, s_in):
  sig_t = jnp.sqrt((1.0 - t_in) ** 2 + t_in**2)
  sig_s = jnp.sqrt((1.0 - s_in) ** 2 + s_in**2)
  dt = t_in - s_in
  small = jnp.abs(dt) < 1e-4
  safe_dt = jnp.where(small, 1.0, dt)
  coeff = jnp.where(
      small,
      (2.0 * t_in - 1.0) / (sig_t**2),
      (1.0 - sig_s / sig_t) / safe_dt,
  )
  return coeff[:, None, None] * x_in, None


v_star, _ = _exact_u(xt_g, t_g, t_g)
imf_v, _, _, _ = flow_map.compute_compound_velocity(
    _exact_u, xt_g, t_g, s_g, v_star
)
mf_v, _, _, _ = flow_map.compute_compound_velocity(
    _exact_u, xt_g, t_g, s_g, x1_g - x0_g
)
imf_jvp_variance = float(jnp.var(imf_v - v_star))
mf_jvp_variance = float(jnp.var(mf_v - v_star))
print(f"MF  compound-velocity JVP variance: {mf_jvp_variance:.4e}")
print(f"iMF compound-velocity JVP variance: {imf_jvp_variance:.4e}")

## 4. Synthetic Class-Conditional 1-D Periodic Dataset

Each class $c \in \lbrace 0, 1, 2 \rbrace$ generates periodic waveforms with a
distinct harmonic frequency and DC offset, paired with Gaussian prior noise
$x_1 \sim \mathcal{N}(0, I)$.


In [ ]:
def make_class_waveforms(num_samples, grid_size, num_classes, seed):
  rng = np.random.default_rng(seed)
  grid = np.linspace(0.0, 2.0 * np.pi, grid_size, endpoint=False)
  labels = rng.integers(0, num_classes, size=(num_samples,), dtype=np.int32)
  phase = rng.uniform(0.0, 2.0 * np.pi, size=(num_samples, 1))
  amp = rng.uniform(0.85, 1.15, size=(num_samples, 1))
  freqs = (labels + 1)[:, None]
  offsets = (labels - 1.0)[:, None] * 1.2
  x_0 = offsets + amp * np.sin(freqs * grid[None, :] + phase)
  x_0 = x_0[..., None].astype(np.float32)
  x_1 = rng.normal(size=x_0.shape).astype(np.float32)
  return {"x_0": x_0, "x_1": x_1, "label": labels}


train_data = make_class_waveforms(
    config.num_train_samples,
    config.grid_size,
    config.num_classes,
    seed=config.seed,
)
eval_data = make_class_waveforms(
    config.num_eval_samples,
    config.grid_size,
    config.num_classes,
    seed=config.seed + 1,
)


def batch_iterator(dataset, batch_size, seed):
  rng = np.random.default_rng(seed)
  n = dataset["x_0"].shape[0]
  while True:
    idx = rng.integers(0, n, size=(batch_size,))
    yield {k: v[idx] for k, v in dataset.items()}

## 5. Train `InContextMeanFlowTransformer` with Auxiliary `v`-Head & Flexible CFG


In [ ]:
net = backbones.InContextMeanFlowTransformer(
    out_channels=config.num_channels,
    hidden_size=config.hidden_size,
    depth=config.depth,
    num_heads=config.num_heads,
    patch_size=config.patch_size,
    aux_head_depth=config.aux_head_depth,
    num_classes=config.num_classes,
    use_cfg_conditioning=True,
    use_interval_conditioning=True,
    residual_scale_init=config.residual_scale_init,
    zero_init_output=config.zero_init_output,
)
imf_model = models.ConditionalImprovedMeanFlowModel(
    input_shape=(config.grid_size, config.num_channels),
    mean_flow_model=net,
    cond_shape={"label": ()},
    num_classes=config.num_classes,
    cfg_mode=config.cfg_mode,
    class_dropout_rate=config.class_dropout_rate,
    cfg_scale_min=config.cfg_scale_min,
    cfg_scale_max=config.cfg_scale_max,
    cfg_scale_beta=config.cfg_scale_beta,
    cfg_t_min_range=config.cfg_t_min_range,
    cfg_t_max_range=config.cfg_t_max_range,
    diagonal_sampling_rate=config.diagonal_sampling_rate,
    norm_power=config.norm_power,
    norm_regularization=config.norm_regularization,
    use_aux_v_head=True,
    aux_v_loss_weight=config.aux_v_loss_weight,
    number_of_eval_steps=config.eval_num_steps,
)
trainer = trainers.ImprovedMeanFlowTrainer(
    ema_decay=config.ema_decay,
    model=imf_model,
    rng=jax.random.PRNGKey(config.seed),
    optimizer=optax.adamw(learning_rate=config.learning_rate),
)

train_iter = batch_iterator(train_data, config.batch_size, seed=config.seed + 2)
train_metrics = trainer.train(train_iter, num_steps=config.num_train_steps)
computed_train = train_metrics.compute()

eval_iter = iter([{k: v[: config.batch_size] for k, v in eval_data.items()}])
computed_eval = trainer.eval(eval_iter, num_steps=1).compute()
print(
    f"Train loss: {float(computed_train['train_loss']):.4f}, "
    f"Eval loss: {float(computed_eval['eval_loss']):.4f}"
)

## 6. 1-NFE CFG Scale & Interval Sweeps (Inference `head='u'` Only)


In [ ]:
infer_fn = trainers.ImprovedMeanFlowTrainer.inference_fn_from_state_dict(
    trainer.train_state,
    net,
    use_ema=True,
    cfg_mode=config.cfg_mode,
    use_aux_v_head=True,
)

x_prior = jnp.asarray(eval_data["x_1"][: config.batch_size])
cond_eval = {"label": jnp.asarray(eval_data["label"][: config.batch_size])}

cfg_sweep_samples = {}
for scale in config.eval_cfg_scales:
  cfg_sweep_samples[scale] = np.asarray(
      flow_map.sample_backward_mean_flow(
          lambda x, t, s, c, w=scale: infer_fn(
              x, t, s, c, cfg_scale=w, cfg_interval=(0.0, 1.0)
          ),
          x_prior,
          num_steps=1,
          cond=cond_eval,
      )
  )

interval_sweep_samples = {}
for interval in config.eval_cfg_intervals:
  interval_sweep_samples[interval] = np.asarray(
      flow_map.sample_backward_mean_flow(
          lambda x, t, s, c, iv=interval: infer_fn(
              x, t, s, c, cfg_scale=2.0, cfg_interval=iv
          ),
          x_prior,
          num_steps=1,
          cond=cond_eval,
      )
  )

metrics = {
    "mf_jvp_variance": mf_jvp_variance,
    "imf_jvp_variance": imf_jvp_variance,
    "final_train_loss": float(computed_train["train_loss"]),
    "eval_loss": float(computed_eval["eval_loss"]),
}
for scale, arr in cfg_sweep_samples.items():
  metrics[f"sample_std_w_{scale:g}"] = float(np.std(arr))

fig, axes = plt.subplots(1, len(config.eval_cfg_scales), figsize=(10, 3))
if len(config.eval_cfg_scales) == 1:
  axes = [axes]
for ax, scale in zip(axes, config.eval_cfg_scales):
  samples = cfg_sweep_samples[scale]
  for cls_idx in range(config.num_classes):
    mask = eval_data["label"][: config.batch_size] == cls_idx
    if np.any(mask):
      ax.plot(samples[mask][0, :, 0], label=f"class {cls_idx}")
  ax.set_title(f"1-NFE (w = {scale:g})")
  ax.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(workdir, "imf_cfg_sweep.png"), dpi=100)
plt.show()